In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../fraud.db")

df = pd.read_sql("SELECT Time, Amount, Class FROM transactions", conn)
df.to_csv("../data/tableau_transactions.csv", index=False)

results = pd.read_sql("SELECT * from model_results", conn)
results.to_csv("../data/teableau_model_results.csv", index=False)

conn.close()

In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../fraud.db")
df = pd.read_sql("SELECT Time, Amount, Class FROM transactions", conn)
conn.close()

# Convert seconds to hours
df['hour'] = (df['Time'] // 3600).astype(int)

# Create time bins of 1 hour each
df['time_bin'] = df['hour'].astype(int)

# Calculate fraud rate per hour bin
fraud_rate = df.groupby('time_bin').agg(
    total=('Class', 'count'),
    fraud_count=('Class', 'sum')
).reset_index()

fraud_rate['fraud_rate'] = fraud_rate['fraud_count'] / fraud_rate['total']

fraud_rate.to_csv('../data/tableau_time.csv', index=False)

In [3]:
confusion_data = pd.DataFrame([
    # Logistic Regression
    {'Model': 'Logistic Regression', 'Actual': 'Legitimate', 'Predicted': 'Legitimate', 'Value': 55478, 'Label': 'True Negative'},
    {'Model': 'Logistic Regression', 'Actual': 'Legitimate', 'Predicted': 'Fraud', 'Value': 1386, 'Label': 'False Positive'},
    {'Model': 'Logistic Regression', 'Actual': 'Fraud', 'Predicted': 'Legitimate', 'Value': 8, 'Label': 'False Negative'},
    {'Model': 'Logistic Regression', 'Actual': 'Fraud', 'Predicted': 'Fraud', 'Value': 90, 'Label': 'True Positive'},
    
    # Random Forest tuned
    {'Model': 'Random Forest (tuned)', 'Actual': 'Legitimate', 'Predicted': 'Legitimate', 'Value': 56857, 'Label': 'True Negative'},
    {'Model': 'Random Forest (tuned)', 'Actual': 'Legitimate', 'Predicted': 'Fraud', 'Value': 7, 'Label': 'False Positive'},
    {'Model': 'Random Forest (tuned)', 'Actual': 'Fraud', 'Predicted': 'Legitimate', 'Value': 15, 'Label': 'False Negative'},
    {'Model': 'Random Forest (tuned)', 'Actual': 'Fraud', 'Predicted': 'Fraud', 'Value': 83, 'Label': 'True Positive'},
])

confusion_data.to_csv('../data/tableau_confusion.csv', index=False)